In [1]:
import torch

In [2]:
x = torch.tensor(3.0, requires_grad= True)
# we want to calculate derivatives

In [3]:
y = x**2

In [4]:
x

tensor(3., requires_grad=True)

In [5]:
y

tensor(9., grad_fn=<PowBackward0>)

In [6]:
y.backward()

In [7]:
x.grad

tensor(6.)

In [8]:
x = torch.tensor(4.0, requires_grad= True)

In [9]:
y = x**2


In [10]:
z = torch.sin(y)

In [11]:
x

tensor(4., requires_grad=True)

In [12]:
y

tensor(16., grad_fn=<PowBackward0>)

In [13]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

In [14]:
z.backward()

In [15]:
x.grad

tensor(-7.6613)

In [16]:
y.grad

/tmp/ipykernel_2094/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


In [17]:
import torch

# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [18]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [19]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [20]:
loss

tensor(6.7012)

In [21]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [22]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [23]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)
# as we have to calculate derivative of w and b

In [24]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [25]:
z = w*x +b

In [26]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [27]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [28]:
loss.backward()

In [29]:
w.grad


tensor(6.6918)

In [30]:
b.grad

tensor(0.9988)

Vector inputs


In [31]:
x = torch.tensor([1.0,2.0,3.0], requires_grad=True)

In [32]:
x

tensor([1., 2., 3.], requires_grad=True)

In [33]:
y = (x**2).mean()

In [34]:
y.backward()

In [35]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

In [36]:
#clearing grad
x = torch.tensor(2.0, requires_grad= True)

In [37]:
y = x**2

In [38]:
y.backward()

In [39]:
x.grad

tensor(4.)

In [40]:
x.grad.zero_()

tensor(0.)

In [41]:
# disable gradient tracking

x = torch.tensor(2.0, requires_grad= True)
x

tensor(2., requires_grad=True)

In [42]:
y = x**2
y

tensor(4., grad_fn=<PowBackward0>)

In [43]:
# option 1 -

x.requires_grad_(False)

tensor(2.)

In [44]:
x

tensor(2.)

In [45]:
y = x**2

In [46]:
y

tensor(4.)

In [47]:
# option 2 - detach
x = torch.tensor(2.0, requires_grad= True)

In [48]:
z = x.detach()
z

tensor(2.)

In [49]:
y = x**2

In [50]:
y

tensor(4., grad_fn=<PowBackward0>)

In [51]:
y1 = z**2
y1

tensor(4.)

In [52]:
y.backward()

In [53]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [54]:
# option 3 - torch.no_grad()
x = torch.tensor(2.0, requires_grad= True)
x

tensor(2., requires_grad=True)

In [55]:
with torch.no_grad():
    y = x**2
    y.backward()
    print(x.grad)

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

# Video 4 Pytorch Training Pipeline

In [56]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [57]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [58]:
df.shape

(569, 33)

In [59]:
df.drop(columns = ['id','Unnamed: 32'], inplace = True)

In [60]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [61]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [62]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [63]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

Numpy arrays to PyTorch Tensors

In [64]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [65]:
X_train_tensor.shape

torch.Size([455, 30])

In [66]:
y_train_tensor.shape

torch.Size([455])

Defining the model

In [67]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
    return loss


In [68]:
learning_rate = 0.1
epochs = 1

In [69]:
#Training pipeline
model = MySimpleNN(X_train_tensor)

# forward pass
for epochs in range(epochs):
  y_pred = model.forward(X_train_tensor)
  loss = model.loss_function(y_pred, y_train_tensor)

  #backward pass
  loss.backward()

#loss calculate

#backwars pass

#parameters update

In [70]:
model.bias

tensor([0.], dtype=torch.float64, requires_grad=True)

Evaluation

In [75]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred =(y_pred > 0.5).float()
print(y_pred)

tensor([[0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
      